# 34 · Workflows: parallelisation, and workflow vs agent
Level L3 · Part 4 · professional · ~30 min · builds on 25, 33 (also uses 26)

## Goal
Run independent model calls at the same time (sectioning), let three checks vote on whether a Scout
summary is accurate (voting), and decide from measured calls, tokens, cost, rounds and accuracy when
this workflow beats an agent doing the same job.

## The idea
Lesson 33 fixed the steps in code and ran them one after another. When steps do **not** depend on each
other, they can run at the same time: that is **parallelisation**. It has two forms. **Sectioning**
splits one task into independent parts (one summary per planet) and joins the results. **Voting** runs
several checks on the *same* thing and combines their answers with a rule, such as "majority wins".
Parallel calls cost the same tokens as sequential ones but finish in the time of the slowest one.
Finally we put the voting workflow next to an agent that checks summaries on its own, and measure both.

Analogy: a newsroom gives each reporter one story (sectioning), then three editors fact-check the
lead story and the majority decides (voting).

## Picture
```mermaid
flowchart LR
    subgraph S["sectioning"]
        T["task"] --> A1["Mars part"] & A2["Venus part"] & A3["..."]
        A1 & A2 & A3 --> J["join"]
    end
    subgraph V["voting"]
        X["one summary"] --> C1["numbers check"] & C2["names check"] & C3["claims check"]
        C1 & C2 & C3 --> M{"majority"}
    end
```

## Step by step
### 1. Start the recorder and set up
The recorder first, as in lesson 33. Without a key the Gemini calls replay from
`../data/llm_cassettes/34_workflows-parallelisation.yaml`.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("34_workflows-parallelisation")

LLM mode: replay (34_workflows-parallelisation.yaml)


Setup from earlier lessons, unchanged: the notes, client and model id (lesson 07), `search_notes`
(lesson 22), and `notes_xml` and `draft_prompt` from lesson 33's chain.

In [2]:
import asyncio, json, os, time
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)

def search_notes(query: str) -> list[dict]:
    """Return the notes whose text contains `query` (ignoring case)."""
    q = query.lower()
    return [note for note in NOTES if q in note["text"].lower()]
print(len(NOTES), "notes | model:", MODEL)

6 notes | model: gemini-3.5-flash-lite


From lesson 33: the notes as XML, and the step-1 draft prompt.

In [3]:
def notes_xml(notes: list[dict]) -> str:
    return "\n".join(f'<note id="{n["id"]}">{n["text"]}</note>' for n in notes)

def draft_prompt(notes: list[dict]) -> str:
    return (f"<notes>\n{notes_xml(notes)}\n</notes>\n\n<instructions>\nSummarise the notes above in at "
            "most 40 words, as plain prose. After each fact, cite its note id in square brackets, like [3]. "
            "Scout links each [id] to the note, so use only ids from the notes above.\n</instructions>")
print(draft_prompt(search_notes("Venus")))

<notes>
<note id="2">Venus spins backwards compared with most planets.</note>
</notes>

<instructions>
Summarise the notes above in at most 40 words, as plain prose. After each fact, cite its note id in square brackets, like [3]. Scout links each [id] to the note, so use only ids from the notes above.
</instructions>


### 2. Why parallel: three slow checks, timed
First without a model. `fake_check` pretends to be a model call that takes half a second and returns
a vote. As in lesson 25, awaiting three of them one by one adds the waits up; `asyncio.gather` overlaps
them.

In [4]:
async def fake_check(name: str, delay: float = 0.5, vote: bool = True) -> bool:
    await asyncio.sleep(delay)                     # stands in for a model call
    return vote

start = time.perf_counter()
one_by_one = [await fake_check("numbers"), await fake_check("names"), await fake_check("claims", vote=False)]
print(f"one by one: {time.perf_counter() - start:.1f} s -> {one_by_one}")

start = time.perf_counter()
together = await asyncio.gather(fake_check("numbers"), fake_check("names"), fake_check("claims", vote=False))
print(f"gather:     {time.perf_counter() - start:.1f} s -> {together}")

one by one: 1.5 s -> [True, True, False]


gather:     0.5 s -> [True, True, False]


Same three votes, a third of the time. Wall-clock time is set by the number of **rounds**: model calls
that must wait for each other. Three parallel calls are one round; three chained calls are three.

### 3. An async `ask` that counts tokens
The real model has an async twin: `client.aio.models.generate_content` takes the same arguments as
lesson 33's call. `ask_async` also adds each reply's token usage to `USAGE`, so we can price every
approach (lesson 12's price for this model).

In [5]:
PRICE = {"input": 0.30, "output": 2.50}           # USD per 1M tokens, gemini-3.5-flash-lite (lesson 12)
USAGE = {"calls": 0, "in": 0, "out": 0}

def count(resp) -> None:
    u = resp.usage_metadata
    USAGE["calls"] += 1
    USAGE["in"] += u.prompt_token_count or 0
    USAGE["out"] += (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)

async def ask_async(prompt: str, schema=None):
    config = types.GenerateContentConfig(temperature=0, max_output_tokens=200, automatic_function_calling=NO_AFC)
    if schema is not None:
        config.response_mime_type, config.response_schema = "application/json", schema
    resp = await client.aio.models.generate_content(model=MODEL, contents=prompt, config=config)
    count(resp)
    return resp.parsed if schema is not None else (resp.text or "").strip()

One helper runs a list of prompts at once and returns the replies in the same order (`gather` keeps
input order). The 50 ms stagger only fixes the order the requests leave in, so the recording replays
each reply to the right prompt; in production you would drop it.

In [6]:
async def parallel(prompts: list[str], schema=None) -> list:
    async def one(i: int, prompt: str):
        await asyncio.sleep(0.05 * i)
        return await ask_async(prompt, schema)
    return await asyncio.gather(*(one(i, p) for i, p in enumerate(prompts)))

def snapshot() -> dict:
    return dict(USAGE)

### 4. Sectioning: one summary per planet, all at once
"Summarise all my notes" splits into independent parts: each planet's notes need no other planet's. So
we send one draft prompt per planet in one round, then join the parts in code.

In [7]:
PLANETS = ["Mars", "Venus", "Jupiter", "Saturn"]
before = snapshot()
parts = await parallel([draft_prompt(search_notes(p)) for p in PLANETS])
report = "\n".join(f"{p}: {text}" for p, text in zip(PLANETS, parts))
print(report)
print("calls:", USAGE["calls"] - before["calls"], "| rounds: 1")

Mars: Mars, which has two moons (Phobos and Deimos) [1], experiences a day lasting about 24 hours and 39 minutes [3]. Additionally, the Perseverance rover successfully landed on the planet in February 2021 [5].
Venus: Unlike most planets, Venus spins in the opposite, backward direction [2].
Jupiter: Jupiter is the largest planet in the solar system [4].
Saturn: Saturn's rings are primarily composed of ice [6].
calls: 4 | rounds: 1


Four calls, but only one round of waiting. Each part cites only its own planet's note ids, because
each call only saw those notes. A chain would have needed four rounds for the same output.

### 5. Voting: three checks on one summary
Now the second form. Each checker looks at the summary from one angle and returns a typed `Vote`.
Lesson 33 had one checker for everything; separate, narrow checks are easier for the model to get right.

In [8]:
class Vote(BaseModel):
    accurate: bool = Field(description="true only if the summary passes this check against the notes")
    problem: str = Field(description="the first problem found, or '' if none")

CHECKS = {
    "numbers": "Check every number, count, date and duration in the summary against the notes.",
    "names":   "Check every name (planet, moon, rover) in the summary against the notes.",
    "claims":  "Check that every claim in the summary is stated in the notes.",
}

def check_prompt(check: str, summary: str, notes: list[dict]) -> str:
    return (f"<notes>\n{notes_xml(notes)}\n</notes>\n\n<summary>{summary}</summary>\n\n"
            f"<instructions>\n{CHECKS[check]} Ignore anything this check is not about.\n</instructions>")

`vote` runs the three checks in one round and applies the rule: the summary is accurate if at least
`need` of the voters say so. The default, 2 of 3, is a majority.

In [9]:
async def vote(summary: str, notes: list[dict], need: int = 2) -> dict:
    votes = await parallel([check_prompt(c, summary, notes) for c in CHECKS], schema=Vote)
    yes = sum(v.accurate for v in votes)
    return {"accurate": yes >= need, "yes": yes,
            "votes": {c: (v.accurate, v.problem) for c, v in zip(CHECKS, votes)}}

Try it on a hand-written summary with one wrong fact: Mars has two moons, not three.

In [10]:
mars_notes = search_notes("Mars")
wrong = "Mars has three moons, Phobos and Deimos [1], and a day there lasts about 24 hours and 39 minutes [3]."
result = await vote(wrong, mars_notes)
print("accurate:", result["accurate"], f"({result['yes']}/3 yes)")
for check, v in result["votes"].items():
    print(f"  {check:8} {v}")

accurate: False (0/3 yes)
  numbers  (False, 'The summary states that Mars has three moons, but the notes state that Mars has two moons.')
  names    (False, 'The summary states Mars has three moons, but note 1 states Mars has two moons.')
  claims   (False, 'The summary states that Mars has three moons, but the notes say it has two small moons, Phobos and Deimos.')


All three said no, each naming the moon count, so the vote is 0 of 3. An error this plain is easy; on
subtler ones the checkers disagree, and then the rule decides. With "2 of 3", one lenient checker cannot
pass a wrong summary on its own.

Now the Mars part that sectioning wrote in step 4.

In [11]:
result = await vote(parts[0], mars_notes)
print(parts[0])
print("accurate:", result["accurate"], f"({result['yes']}/3 yes)", {c: v[0] for c, v in result["votes"].items()})

Mars, which has two moons (Phobos and Deimos) [1], experiences a day lasting about 24 hours and 39 minutes [3]. Additionally, the Perseverance rover successfully landed on the planet in February 2021 [5].
accurate: True (3/3 yes) {'numbers': True, 'names': True, 'claims': True}


### 6. The same job as an agent
Could an agent do this instead? Give it `search_notes` and let it decide what to look up. This is
lesson 26's agent loop, trimmed (with `to_contents` and the Gemini wrapper, now counting tokens). If you
skipped it, read it as: call the model, run the tools it asks for, stop when it answers or at max_steps.

In [12]:
SEARCH_NOTES_SCHEMA = {
    "name": "search_notes",
    "description": "Search the user's notes. Returns every note whose text contains the query word "
                   "(case-insensitive), as a list of {id, text}.",
    "parameters": {"type": "object", "required": ["query"],
                   "properties": {"query": {"type": "string", "description": "one word, e.g. 'Mars'"}}},
}

def to_contents(messages: list[dict]) -> list[types.Content]:
    contents = []
    for m in messages:
        if m["role"] == "user":
            contents.append(types.Content(role="user", parts=[types.Part(text=m["text"])]))
        elif m["role"] == "assistant":
            contents.append(m["raw"])
        else:
            part = types.Part(function_response=types.FunctionResponse(
                id=m["call_id"], name=m["name"], response={"result": json.loads(m["content"])}))
            if contents[-1].role == "user" and contents[-1].parts[0].function_response:
                contents[-1].parts.append(part)
            else:
                contents.append(types.Content(role="user", parts=[part]))
    return contents

The wrapper and the loop, as in lesson 26, with `count(resp)` added and the per-step prints removed.

In [13]:
AGENT_SYSTEM = ("You are Scout. The user gives you a summary of their notes. Search the notes to check it. "
                "Reply with ACCURATE or INACCURATE as the first word, then one short reason.")

class GeminiModel:
    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        decls = [types.FunctionDeclaration(name=t["name"], description=t["description"],
                                           parameters_json_schema=t["parameters"]) for t in tools]
        cfg = types.GenerateContentConfig(system_instruction=AGENT_SYSTEM, max_output_tokens=300, temperature=0,
                                          tools=[types.Tool(function_declarations=decls)],
                                          automatic_function_calling=NO_AFC)
        resp = client.models.generate_content(model=MODEL, contents=to_contents(messages), config=cfg)
        count(resp)
        calls = [{"id": fc.id, "name": fc.name, "args": dict(fc.args)} for fc in resp.function_calls or []]
        text = "" if calls else (resp.text or "")
        return {"role": "assistant", "tool_calls": calls, "text": text, "raw": resp.candidates[0].content}

The loop: call the model, run every tool it asks for, stop when it answers or at `max_steps`.

In [14]:
def run_agent(question: str, model, tools: dict, schemas: list[dict], max_steps: int = 6) -> dict:
    messages = [{"role": "user", "text": question}]
    for step in range(1, max_steps + 1):
        reply = model(messages, schemas)
        messages.append(reply)
        if not reply["tool_calls"]:
            return {"answer": reply["text"] or "", "stop": "end_of_turn", "steps": step}
        for call in reply["tool_calls"]:
            result = tools[call["name"]](**call["args"])
            messages.append({"role": "tool", "call_id": call["id"], "name": call["name"],
                             "content": json.dumps(result)})
    return {"answer": "", "stop": "step_limit", "steps": max_steps}

### 7. Measure both on a small labelled set
Four summaries we wrote by hand, with the right answer. Two are accurate; two have one wrong fact (a
moon count, a rover name). Same set, same model, for both approaches.

In [15]:
CASES = [
    ("Mars has two small moons, Phobos and Deimos [1], and a day there lasts about 24 hours and 39 minutes [3].", True),
    ("The rover Perseverance landed on Mars in February 2021 [5]. Mars has two moons [1].", True),
    ("Mars has three moons [1] and a day there lasts about 24 hours and 39 minutes [3].", False),
    ("The rover Curiosity landed on Mars in February 2021 [5].", False),
]

def cost(u: dict) -> float:
    return (u["in"] * PRICE["input"] + u["out"] * PRICE["output"]) / 1e6

def summary_row(name: str, before: dict, correct: int, rounds: int) -> dict:
    u = {k: USAGE[k] - before[k] for k in USAGE}
    return {"approach": name, "correct": f"{correct}/{len(CASES)}", "calls": u["calls"], "rounds": rounds,
            "tokens_in": u["in"], "tokens_out": u["out"], "usd": round(cost(u), 6)}

The workflow: code searches the notes (it knows the topic is Mars), then three checks vote in one round
per case.

In [16]:
before, correct = snapshot(), 0
for summary, label in CASES:
    r = await vote(summary, mars_notes)
    correct += r["accurate"] == label
    print(f"label {label!s:5} | workflow says {r['accurate']!s:5} ({r['yes']}/3)")
workflow_row = summary_row("workflow (3 votes)", before, correct, rounds=len(CASES))

label True  | workflow says True  (3/3)


label True  | workflow says True  (3/3)


label False | workflow says False (0/3)


label False | workflow says False (0/3)


The agent: same four summaries, and it decides itself what to search. Every step is a round, because
each model call waits for the previous one.

In [17]:
before, correct, agent_rounds = snapshot(), 0, 0
for summary, label in CASES:
    run = run_agent(f"Check this summary: {summary}", GeminiModel(), {"search_notes": search_notes},
                    [SEARCH_NOTES_SCHEMA])
    says = run["answer"].strip().upper().startswith("ACCURATE")
    correct, agent_rounds = correct + (says == label), agent_rounds + run["steps"]
    print(f"label {label!s:5} | agent says {says!s:5} in {run['steps']} steps: {run['answer'][:60]!r}")
agent_row = summary_row("agent (search tool)", before, correct, rounds=agent_rounds)

label True  | agent says True  in 2 steps: 'ACCURATE Both facts are directly supported by notes 1 and 3.'
label True  | agent says True  in 2 steps: 'ACCURATE Both statements are directly supported by the notes'
label False | agent says False in 2 steps: 'INACCURATE Mars has two moons, not three.'
label False | agent says False in 3 steps: 'INACCURATE The rover Perseverance, not Curiosity, landed on '


Side by side. `rounds` is what you wait for: wall time is roughly rounds times the time of one call.
(Replayed calls return instantly, so we count rounds instead of timing them; in live mode you can
time them. The 1.5 s vs 0.5 s in step 2 is the only timing here, and it uses a fake call.)

In [18]:
for row in (workflow_row, agent_row):
    print(row)

{'approach': 'workflow (3 votes)', 'correct': '4/4', 'calls': 12, 'rounds': 4, 'tokens_in': 2872, 'tokens_out': 294, 'usd': 0.001597}
{'approach': 'agent (search tool)', 'correct': '4/4', 'calls': 9, 'rounds': 9, 'tokens_in': 1443, 'tokens_out': 150, 'usd': 0.000808}


Both got 4 of 4 right. The workflow made 12 calls and cost about twice as much (three checkers each read
the notes), but waited only 4 rounds, one per summary, every time. The agent made 9 calls for half the
cost, but every call was its own round (9), and the step count was the model's choice (2 or 3). So the
workflow wins on rounds and predictability here, the agent on cost. The workflow got the Mars notes
from our code; the agent had to find them, so this is a fair test only because the topic was known.
Four cases cannot separate their accuracy; that needs a bigger labelled set (lesson 48).

### 8. When a workflow beats an agent
The workflow won its speed because **we knew the steps**: the topic was Mars, so code did the search.
If the summary could be about anything, or checking needed steps we cannot list in advance, the agent's
flexibility is worth its extra rounds.

A short guide:

| Question | Workflow | Agent |
|---|---|---|
| Can you list the steps in advance? | yes | no |
| Latency | fixed rounds; parallel parts cost one round | one round per step, number varies |
| Cost | fixed calls per request (voting multiplies it) | varies with the steps the model takes |
| Testing | every path known | needs evals over many runs |

## At work
- **Sectioning as a guardrail.** Anthropic's "Building effective agents" uses sectioning to run a
  screening call for unsafe content beside the call that answers, and to score each aspect of an eval
  in its own call: "LLMs generally perform better when each consideration is handled by a separate LLM
  call". → lesson 48
- **Voting for high-stakes checks.** The same article votes with several prompts on code vulnerability
  reviews and content moderation, and tunes the vote threshold to trade false alarms for misses. Voting
  multiplies tokens, so teams vote only where a miss is expensive. → lesson 54
- **Parallel calls share one rate limit.** Five parallel calls per request use five times the
  requests-per-minute; under load they hit 429s together, so production code caps concurrency.
  → lesson 38
- **"Agentic systems often trade latency and cost for better task performance"** (Anthropic): teams
  measure both versions on the same set before picking the agent. → lesson 45

## What just happened
- **Sectioning** split "summarise my notes" into four independent per-planet calls, run in one round
  with `asyncio.gather` on `client.aio`, then joined in code.
- **Voting** ran three narrow checks (numbers, names, claims) on one summary in one round and combined
  them with a threshold, 2 of 3 by default.
- We counted calls, tokens, cost and rounds for the voting workflow and for a tool-using agent on the
  same four labelled summaries.
- Both were 4 of 4 correct. The workflow cost about twice as much but waited 4 fixed rounds; the agent
  was cheaper but took 9 rounds of its own choosing. Which wins depends on whether you can list the
  steps in advance and whether latency or cost matters more.

## Common mistakes
**1. Passing a list to `gather`.** `gather` takes the awaitables as separate arguments. Given one list,
it fails before anything runs, with an error that does not mention lists of coroutines at all.

In [19]:
checks = [fake_check("numbers"), fake_check("names"), fake_check("claims")]
votes = await asyncio.gather(checks)

TypeError: unhashable type: 'list'

Fix: unpack the list with `*`. (The failed call above left three coroutines that never ran; we make
fresh ones.)

In [20]:
for c in checks:
    c.close()
votes = await asyncio.gather(*[fake_check("numbers"), fake_check("names"), fake_check("claims")])
print(votes)

[True, True, True]


**2. One failing voter sinks the vote.** If any checker raises (a timeout, a 503), `gather` raises
too, and the two good votes are lost.

In [21]:
async def broken_check(name: str) -> bool:
    await asyncio.sleep(0.1)
    raise TimeoutError(f"{name} check timed out")

Two good checkers and one broken one, gathered as usual:

In [22]:
votes = await asyncio.gather(fake_check("numbers"), broken_check("names"), fake_check("claims", vote=False))

TimeoutError: names check timed out

Fix: `return_exceptions=True` returns the error in that voter's slot instead of raising. Count it as
an abstention and decide from the votes you have, and log it, so a checker that keeps failing is seen.

In [23]:
votes = await asyncio.gather(fake_check("numbers"), broken_check("names"), fake_check("claims", vote=False),
                             return_exceptions=True)
valid = [v for v in votes if isinstance(v, bool)]
print("abstained:", [str(v) for v in votes if isinstance(v, Exception)], "| yes:", sum(valid), "of", len(valid))

abstained: ['names check timed out'] | yes: 1 of 2


## Try it
Write `vote_with_timeout(checks, timeout, need)`. `checks` is a list of coroutines like `fake_check(...)`.
New tool: `await asyncio.wait_for(coro, timeout)` raises `asyncio.TimeoutError` if `coro` takes longer
than `timeout` seconds. Wrap each check in it and combine with `asyncio.gather(..., return_exceptions=True)`
so a check that times out or raises comes back as an exception: count it as abstained. Return
`{"accurate": yes >= need, "yes": ..., "abstained": ...}`. Test it with three checks where one has
`delay=2.0` and `timeout=1.0`: it should finish in about 1 s.

In [24]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [25]:
llm_replay.stop()

## Key terms
- **parallelisation** — running model calls that do not depend on each other at the same time, so they finish in one round.
- **sectioning** — splitting one task into independent parts, running them in parallel, and joining the results in code.
- **voting** — running several checks or attempts on the same input and combining their answers with a rule such as a majority.
- **vote threshold** — how many voters must agree (for example 2 of 3); a higher threshold means fewer false passes and more false alarms.
- **round** — a group of model calls that wait for nothing but each other; wall time grows with rounds, not with calls.